# EDA: deposit level and pre-DRS recycling rate

Univariate view of both axes (plastic recycling rate, deposit per country), then two scatter plots:
how much plastic recycling changed after each DRS launch, against (A) the deposit and (B) the recycling rate before the DRS (our heterogeneity variable).

Uses the outputs of `build_drs_dataset_plastic_v2.ipynb` in `data/`, like `eda.ipynb`. All figures are saved in `figures/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR, FIG_DIR = Path("data"), Path("figures")
FIG_DIR.mkdir(exist_ok=True)

panel = pd.read_csv(DATA_DIR / "drs_recycling_panel_plastic.csv")
snap = pd.read_csv(DATA_DIR / "drs_deposit_snapshots_plastic.csv")

ADOPT, NEVER, BEFORE = "2_drs_introduced_in_window", "3_no_drs_in_window", "1_drs_before_window"
METHOD_BREAK = 2020   # Eurostat moved the point where recycling is measured from reference year 2020
BLUE, ORANGE, GREY, INK, INK2, GRID = "#2a78d6", "#eb6834", "#8a8984", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlelocation": "left"})
print("panel:", panel.shape, "| snapshots:", snap.shape)

## 1. Change in plastic recycling around each DRS launch

For every country that introduced a DRS inside the data window:

**change** = mean rate in the first 2 DRS years − mean rate in the 2 years before,
minus the same change in the countries without a DRS (same calendar years, only countries with data in all those years).

Each window stays on one side of the 2020 calculation change, so the method break never falls inside a comparison.

In [ ]:
rates = panel.pivot_table(index="year", columns="geo", values="plastic_recycling_rate")
never = [g for g in panel.loc[panel["drs_group"] == NEVER, "geo"].unique() if g in rates.columns]
starts = (panel.loc[panel["drs_group"] == ADOPT].groupby("geo")["plastic_drs_start"].first()
               .dropna().astype(int))
last_year = int(rates.index.max())

rows = []
for geo, s in starts.items():
    pre = [s - 2, s - 1]
    post = [y for y in (s, s + 1) if y <= last_year]
    if s < METHOD_BREAK:
        post = [y for y in post if y < METHOD_BREAK]
    else:
        pre = [y for y in pre if y >= METHOD_BREAK]
    yrs = pre + post
    if geo not in rates.columns or not pre or not post or rates.loc[yrs, geo].isna().any():
        continue
    ctrl = [c for c in never if rates.loc[yrs, c].notna().all()]
    own = rates.loc[post, geo].mean() - rates.loc[pre, geo].mean()
    comp = (rates.loc[post, ctrl].mean() - rates.loc[pre, ctrl].mean()).mean()
    rows.append({"geo": geo, "drs_start": s, "years_before": f"{pre[0]}-{pre[-1]}",
                 "years_after": f"{post[0]}-{post[-1]}" if len(post) > 1 else str(post[0]), "rate_before": rates.loc[pre, geo].mean(),
                 "own_change": own, "comparison_change": comp, "n_comparison": len(ctrl),
                 "did": own - comp})

deposit = (snap.dropna(subset=["deposit_max_eur"]).query("deposit_max_eur > 0")
               .sort_values("snapshot_year").groupby("geo")["deposit_max_eur"].last())
did = pd.DataFrame(rows).set_index("geo").join(deposit.rename("deposit_eur"))
did.round(2)

## 2. Univariate: the two axes

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.2, 1]})

# A. Outcome: plastic recycling rate, all country-years, split by calculation method
bins = np.arange(0, 85, 5)
for label, sel, colour in [(f"before {METHOD_BREAK} (old method)", panel["year"] < METHOD_BREAK, BLUE),
                           (f"{METHOD_BREAK} onwards (new method)", panel["year"] >= METHOD_BREAK, ORANGE)]:
    v = panel.loc[sel, "plastic_recycling_rate"].dropna()
    ax1.hist(v, bins=bins, density=True, histtype="step", lw=2.2, color=colour, label=f"{label}, n={len(v)}")
ax1.set(xlabel="Plastic packaging recycling rate, %", ylabel="Share of country-years (density)",
        title="A. Plastic recycling rate across country-years")
ax1.grid(axis="y", color=GRID)
ax1.legend(frameon=False, loc="upper right")

# B. Deposit per country (latest snapshot); a dot per country is clearer than a histogram for ~20 values
dep = (deposit.to_frame("eur").join(panel.drop_duplicates("geo").set_index("geo")[["country", "drs_group"]])
              .sort_values("eur"))
colours = dep["drs_group"].map({ADOPT: BLUE, BEFORE: GREY}).fillna(GREY)
ax2.hlines(range(len(dep)), 0, dep["eur"], color=GRID, lw=1)
ax2.scatter(dep["eur"], range(len(dep)), color=colours, s=55, zorder=3, edgecolor="white", linewidth=1.2)
for i, v in enumerate(dep["eur"]):
    ax2.text(v + 0.008, i, f"€{v:.2f}", va="center", fontsize=8.5, color=INK2)
ax2.set_yticks(range(len(dep)), dep["country"])
ax2.set_xlim(0, dep["eur"].max() * 1.25)
ax2.set(xlabel="Highest deposit per container, EUR (latest available)",
        title="B. Deposit level by country")
ax2.scatter([], [], color=BLUE, s=55, label="DRS introduced 2001-2023")
ax2.scatter([], [], color=GREY, s=55, label="DRS before 2000 / other")
ax2.legend(frameon=False, loc="lower right")

fig.suptitle("The two axes on their own", fontsize=14, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIG_DIR / "univariate_rate_and_deposit.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Scatter: did the DRS add more where the deposit was higher, or where recycling was weak?

One dot per adopter. With about 10 countries this is a description, not a test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
panels = [("deposit_eur", "Highest deposit per container, EUR", "A. Change vs deposit level"),
          ("rate_before", "Plastic recycling rate in the 2 years before the DRS, %",
           "B. Change vs recycling rate before the DRS")]
for ax, (col, xlabel, title) in zip(axes, panels):
    d = did.dropna(subset=[col, "did"])
    ax.axhline(0, color=GREY, lw=1)
    ax.scatter(d[col], d["did"], s=70, color=BLUE, edgecolor="white", linewidth=1.5, zorder=3)
    for geo, r in d.iterrows():
        ax.annotate(f"{geo} {r['drs_start']}", (r[col], r["did"]), xytext=(6, 4),
                    textcoords="offset points", fontsize=8.5, color=INK2)
    if len(d) >= 3:   # simple descriptive slope, not a causal estimate
        b, a = np.polyfit(d[col], d["did"], 1)
        xs = np.linspace(d[col].min(), d[col].max(), 50)
        ax.plot(xs, a + b * xs, color=ORANGE, lw=1.5, label=f"linear fit, slope {b:+.2f}")
        ax.legend(frameon=False, loc="best")
    ax.set_ylim(min(0, d["did"].min()) - 1, d["did"].max() * 1.15 + 1)
    ax.set(xlabel=xlabel, title=f"{title} (n = {len(d)})")
    ax.grid(color=GRID)
axes[0].set_ylabel("Change after DRS minus change in no-DRS countries, pp")
fig.suptitle("How much did plastic recycling change after each DRS launch?", fontsize=14,
             fontweight="bold", x=0.01, ha="left")
fig.text(0.01, -0.02, "Change = 2 years after vs 2 years before launch, net of the same change in countries "
         "without a DRS. Windows never cross the 2020 calculation change.", fontsize=8.5, color=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIG_DIR / "scatter_change_vs_deposit_and_baseline.png", dpi=200, bbox_inches="tight")
plt.show()
did[["drs_start", "deposit_eur", "rate_before", "did"]].round(2).sort_values("did")

**Check before presenting:** the start year comes from `plastic_drs_start`. Malta launched on 14 Nov 2022 and Romania on 30 Nov 2023, so a start year of 2022 or 2023 counts a year with only about 6 weeks of DRS as "treated". Coding the first year with at least 6 months of operation (Malta 2023, Romania 2024) is easier to defend; Romania then has no post-DRS year yet and drops out.